In [9]:
import pandas as pd
from pathlib import Path

BASE = Path(r'B:\Semester 4 UU\thesis-best-paper-trajectories')
DATA_MATCHED = BASE / 'data' / 'matched'
DATA_DERIVED = BASE / 'data' / 'derived'

DATA_DERIVED.mkdir(parents=True, exist_ok=True)

EDGES_PATH = DATA_MATCHED / 'award_to_citing_edges.csv'
CITING_PATH = DATA_MATCHED / 'citing_papers.csv'
JUNIOR_PATH = DATA_MATCHED / 'junior_authors_all_conferences.csv'

OUT_PATH = DATA_DERIVED / 'junior_awardpaper_self_cites_summary.csv'

print("Paths set.")

Paths set.


In [16]:
edges = pd.read_csv(EDGES_PATH)
citing = pd.read_csv(CITING_PATH)
junior = pd.read_csv(JUNIOR_PATH)

# Rename columns to align with rest of pipeline
# Use work_id (OpenAlex work) as award_paper_id, not the numeric award_id
junior = junior.rename(columns={
    'work_id': 'award_paper_id',
    'author_id': 'junior_author_id',
    'author_name': 'junior_author_name',
    'conference': 'award_conference',
})

print("Edges:", edges.shape)
print("Citing:", citing.shape)
print("Junior mapping:", junior.shape)

edges.head(3)

C:\Users\sla99\AppData\Local\Temp\ipykernel_23848\1787545595.py:2: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  citing = pd.read_csv(CITING_PATH)


Edges: (507718, 4)
Citing: (438807, 23)
Junior mapping: (603, 14)


,award_paper_id,award_year,award_conference,citing_paper_id
0,https://openalex.org/W2788603415,2018,AAAI,https://openalex.org/W2962990649
1,https://openalex.org/W2788603415,2018,AAAI,https://openalex.org/W3150570724
2,https://openalex.org/W2788603415,2018,AAAI,https://openalex.org/W3011736398


In [17]:
print("Edges columns:", edges.columns.tolist())
print("Citing columns:", citing.columns.tolist())
print("Junior columns:", junior.columns.tolist())

edges['award_paper_id'] = edges['award_paper_id'].astype(str).str.strip()
edges['citing_paper_id'] = edges['citing_paper_id'].astype(str).str.strip()
citing['openalex_id'] = citing['openalex_id'].astype(str).str.strip()
junior['award_paper_id'] = junior['award_paper_id'].astype(str).str.strip()
junior['junior_author_id'] = junior['junior_author_id'].astype(str).str.strip()

Edges columns: ['award_paper_id', 'award_year', 'award_conference', 'citing_paper_id']
Citing columns: ['openalex_id', 'doi', 'title', 'publication_year', 'publication_date', 'type', 'cited_by_count', 'is_retracted', 'is_oa', 'source_id', 'source_name', 'source_type', 'source_issn', 'author_ids', 'author_names', 'author_positions', 'author_count', 'first_author_institution', 'top_topic', 'top_field', 'counts_by_year', 'cites_award_papers', 'cites_n_award_papers']
Junior columns: ['award_id', 'award_conference', 'award_year', 'award_paper_id', 'award_title', 'junior_author_id', 'junior_author_name', 'author_position', 'career_age_at_award', 'total_pubs_at_award', 'institutions', 'is_corresponding', 'match_route', 'core_rank']


In [18]:
# Make junior award_paper_id match the OpenAlex URL format used in edges
mask_short = ~junior['award_paper_id'].str.startswith('http')
junior.loc[mask_short, 'award_paper_id'] = (
    'https://openalex.org/' + junior.loc[mask_short, 'award_paper_id']
)

In [19]:
# Step 2 — Merge edges with citing metadata

merged = edges.merge(
    citing,
    left_on='citing_paper_id',
    right_on='openalex_id',
    how='left'
)

print("Merged edges + citing:", merged.shape)
merged[['award_paper_id', 'citing_paper_id', 'award_year',
        'publication_year', 'publication_date', 'author_ids']].head(5)

Merged edges + citing: (507718, 27)


,award_paper_id,citing_paper_id,award_year,publication_year,publication_date,author_ids
0,https://openalex.org/W2788603415,https://openalex.org/W2962990649,2018,2019.0,2019-06-01,https://openalex.org/A5078487642|https://opena...
1,https://openalex.org/W2788603415,https://openalex.org/W3150570724,2018,2019.0,2019-02-22,https://openalex.org/A5100424963
2,https://openalex.org/W2788603415,https://openalex.org/W3011736398,2018,2019.0,2019-12-01,https://openalex.org/A5111488071|https://opena...
3,https://openalex.org/W2788603415,https://openalex.org/W4394615984,2018,2024.0,2024-04-09,https://openalex.org/A5001195684|https://opena...
4,https://openalex.org/W2788603415,https://openalex.org/W2970634945,2018,2019.0,2019-01-01,https://openalex.org/A5026521600|https://opena...


In [28]:
# Step 3 — Filter to post-award citing papers (year-based)

merged['publication_year'] = pd.to_numeric(
    merged['publication_year'], errors='coerce'
)
merged['award_year'] = pd.to_numeric(
    merged['award_year'], errors='coerce'
)

pre_rows = len(merged)
merged_post = merged[merged['publication_year'] > merged['award_year']].copy()
post_rows = len(merged_post)

print(f"Total citing edges: {pre_rows:,}")
print(f"Post-award citing edges: {post_rows:,}")
print("Year ranges:")
print("  award_year:", merged_post['award_year'].min(), "–", merged_post['award_year'].max())
print("  citing_year:", merged_post['publication_year'].min(), "–", merged_post['publication_year'].max())

Total citing edges: 507,718
Post-award citing edges: 502,992
Year ranges:
  award_year: 2000 – 2018
  citing_year: 2001.0 – 2026.0


In [29]:
# Step 4 — Join in junior authors

cols_keep = [
    'award_paper_id',
    'award_year',
    'award_conference',
    'citing_paper_id',
    'publication_year',
    'publication_date',
    'author_ids',
]
merged_post = merged_post[cols_keep].copy()

merged_junior = merged_post.merge(
    junior,
    on='award_paper_id',
    how='inner'
)

print("Merged with junior authors:", merged_junior.shape)
print(merged_junior.columns.tolist())

# Consolidate award_year and award_conference into single columns
merged_junior['award_year'] = merged_junior['award_year_x'].fillna(
    merged_junior['award_year_y']
)
merged_junior['award_conference'] = merged_junior['award_conference_x'].fillna(
    merged_junior['award_conference_y']
)

# Drop the old suffixed columns to avoid confusion
merged_junior = merged_junior.drop(
    columns=['award_year_x', 'award_year_y',
             'award_conference_x', 'award_conference_y'],
    errors='ignore'
)

print("Merged with junior authors:", merged_junior.shape)
merged_junior[['award_paper_id', 'junior_author_id', 'citing_paper_id',
               'publication_year', 'author_ids']].head(5)

Merged with junior authors: (176114, 20)
['award_paper_id', 'award_year_x', 'award_conference_x', 'citing_paper_id', 'publication_year', 'publication_date', 'author_ids', 'award_id', 'award_conference_y', 'award_year_y', 'award_title', 'junior_author_id', 'junior_author_name', 'author_position', 'career_age_at_award', 'total_pubs_at_award', 'institutions', 'is_corresponding', 'match_route', 'core_rank']
Merged with junior authors: (176114, 18)


,award_paper_id,junior_author_id,citing_paper_id,publication_year,author_ids
0,https://openalex.org/W2788603415,https://openalex.org/A5014823249,https://openalex.org/W2962990649,2019.0,https://openalex.org/A5078487642|https://opena...
1,https://openalex.org/W2788603415,https://openalex.org/A5014823249,https://openalex.org/W3150570724,2019.0,https://openalex.org/A5100424963
2,https://openalex.org/W2788603415,https://openalex.org/A5014823249,https://openalex.org/W3011736398,2019.0,https://openalex.org/A5111488071|https://opena...
3,https://openalex.org/W2788603415,https://openalex.org/A5014823249,https://openalex.org/W4394615984,2024.0,https://openalex.org/A5001195684|https://opena...
4,https://openalex.org/W2788603415,https://openalex.org/A5014823249,https://openalex.org/W2970634945,2019.0,https://openalex.org/A5026521600|https://opena...


In [30]:
merged_junior['author_ids'] = merged_junior['author_ids'].fillna('').astype(str)

def has_author(author_ids_str, author_id):
    if not author_ids_str or not author_id:
        return False
    ids = author_ids_str.split('|')
    return author_id in ids

merged_junior['is_self_citing_awardpaper_by_junior'] = merged_junior.apply(
    lambda row: has_author(row['author_ids'], row['junior_author_id']),
    axis=1
).astype(int)

In [27]:
print(merged_junior.columns.tolist())

['award_paper_id', 'award_year_x', 'award_conference_x', 'citing_paper_id', 'publication_year', 'publication_date', 'author_ids', 'award_id', 'award_conference_y', 'award_year_y', 'award_title', 'junior_author_id', 'junior_author_name', 'author_position', 'career_age_at_award', 'total_pubs_at_award', 'institutions', 'is_corresponding', 'match_route', 'core_rank', 'is_self_citing_awardpaper_by_junior']


In [31]:
# Step 6 — Aggregate to author–award level

award_cite_counts = (
    merged_post
    .groupby('award_paper_id')['citing_paper_id']
    .nunique()
    .reset_index()
    .rename(columns={'citing_paper_id': 'n_citing_papers_total_post_award'})
)

author_self_counts = (
    merged_junior
    .groupby(['award_paper_id', 'junior_author_id'], as_index=False)
    .agg(
        n_citing_papers_with_junior_author_post_award=(
            'is_self_citing_awardpaper_by_junior', 'sum'
        ),
        n_distinct_citing_papers_with_junior=('citing_paper_id', 'nunique'),
        award_year=('award_year', 'first'),
        award_conference=('award_conference', 'first'),
        junior_author_name=('junior_author_name', 'first'),
    )
)

author_summary = author_self_counts.merge(
    award_cite_counts,
    on='award_paper_id',
    how='left'
)

def _share(row):
    total = row['n_citing_papers_total_post_award']
    if not total or total <= 0:
        return 0.0
    return row['n_citing_papers_with_junior_author_post_award'] / total

author_summary['share_self_citing'] = author_summary.apply(_share, axis=1)

author_summary.head(10)

,award_paper_id,junior_author_id,n_citing_papers_with_junior_author_post_award,n_distinct_citing_papers_with_junior,award_year,award_conference,junior_author_name,n_citing_papers_total_post_award,share_self_citing
0,https://openalex.org/W108223363,https://openalex.org/A5089862531,0,23,2013,ICML,Roi Livni,23,0.000000
1,https://openalex.org/W110738662,https://openalex.org/A5102785744,0,44,2007,IJCAI,Maleeha Qazi,44,0.000000
2,https://openalex.org/W118378920,https://openalex.org/A5012667159,0,24,2006,SIGMOD,Timothy David Brody,24,0.000000
3,https://openalex.org/W146360823,https://openalex.org/A5017775499,0,18,2004,OSDI,Paul Twohey,18,0.000000
4,https://openalex.org/W146360823,https://openalex.org/A5056288677,0,18,2004,OSDI,Junfeng Yang,18,0.000000
5,https://openalex.org/W1480355964,https://openalex.org/A5081042997,1,62,2010,SODA,Arash Asadpour,62,0.016129
6,https://openalex.org/W1493893823,https://openalex.org/A5060832606,8,702,2008,OSDI,Dennis Fetterly,702,0.011396
7,https://openalex.org/W1512874001,https://openalex.org/A5007903730,1,120,2012,AAAI,Zhanying He,120,0.008333
8,https://openalex.org/W1515884010,https://openalex.org/A5030863830,0,2,2010,ICML,Nishant A. Mehta,2,0.000000
9,https://openalex.org/W1538864647,https://openalex.org/A5028755343,1,53,2010,AAAI,Ruoyun Huang,53,0.018868


In [32]:
# Step 7 — Save to data/derived

author_summary.to_csv(OUT_PATH, index=False)

print("Saved author-level summary to:")
print(OUT_PATH)
print("Shape:", author_summary.shape)

author_summary.describe(include='all').T.head(15)

Saved author-level summary to:
B:\Semester 4 UU\thesis-best-paper-trajectories\data\derived\junior_awardpaper_self_cites_summary.csv
Shape: (596, 9)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
award_paper_id,596,463,https://openalex.org/W2116907335,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
junior_author_id,596,596,https://openalex.org/A5089862531,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_citing_papers_with_junior_author_post_award,596.0,NaN,NaN,NaN,3.149329,5.100949,0.0,0.0,1.0,4.0,45.0
n_distinct_citing_papers_with_junior,596.0,NaN,NaN,NaN,295.493289,1311.541294,1.0,46.0,102.0,203.25,28275.0
award_year,596.0,NaN,NaN,NaN,2011.493289,4.984509,2000.0,2008.0,2012.5,2016.0,2018.0
award_conference,596,30,CHI,119,NaN,NaN,NaN,NaN,NaN,NaN,NaN
junior_author_name,596,596,Roi Livni,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_citing_papers_total_post_award,596.0,NaN,NaN,NaN,295.493289,1311.541294,1.0,46.0,102.0,203.25,28275.0
share_self_citing,596.0,NaN,NaN,NaN,0.036377,0.068801,0.0,0.0,0.008033,0.046067,0.5
